# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Fetch and print a brief description
meta_obj = dataset.metadata
print(f"{meta_obj.name}: {meta_obj.description[:200]}...")

## 2. Data Overview
List available record sets, their fields, and column information, referencing all entities using their `@id` fields.

We'll enumerate all record sets, then for each, enumerate their fields and columns.

In [ ]:
# List all available record sets, fields, and columns with their @id
record_set_ids = []

print("\nAvailable Record Sets:")
for rs in dataset.metadata.record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    Name: {rs.name}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        print("    Fields:")
        for field in rs.fields:
            print(f"      Field @id: {field.id}, name: {field.name}")
            if hasattr(field, 'columns'):
                print("        Columns:")
                for col in field.columns:
                    print(f"          Column @id: {col.id}, name: {col.name}, dataType: {col.data_type if hasattr(col, 'data_type') else 'N/A'}")

## 3. Data Extraction

Now, we'll load the records of each record set and extract them into pandas DataFrames. The variables below reference all entities by their `@id`, in line with best practice for repeatable FAIR data wrangling.

We'll demonstrate exploratory analysis for one record set as an example and list the columns present.

In [ ]:
# Load data from each record set into a DataFrame
dataframes = {}
print("\nExtracting data for each record set:\n")

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"RecordSet: {record_set_id} --> Columns: {dataframes[record_set_id].columns.tolist()}")
        if not dataframes[record_set_id].empty:
            print(dataframes[record_set_id].head(2))
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")

# Use the first record set for additional analysis below
primary_record_set_id = record_set_ids[0] if len(record_set_ids) else None

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and basic grouping to a numeric field. **All field references use the `@id` value that uniquely identifies them.**

Let's inspect the available columns and select a suitable numeric field for demonstration.

In [ ]:
# Show columns and find a candidate numeric field
if primary_record_set_id:
    cols = dataframes[primary_record_set_id].columns.tolist()
    print(f"Available columns in {primary_record_set_id}:\n", cols)

    # Heuristically select a numeric field for demo (feel free to adjust the name below to match real data)
    sample_numeric_fields = [c for c in cols if 'abundance' in c.lower() or 'mw' in c.lower() or 'coverage' in c.lower() or 'count' in c.lower() or 'number' in c.lower() or 'peptide' in c.lower() or 'intensity' in c.lower()]
    if sample_numeric_fields:
        numeric_field_id = sample_numeric_fields[0]  # using field @id (column name)
        print(f"Using '{numeric_field_id}' as the numeric field for demonstration.")

        # Remove NAs and do filtering/normalization
        df = dataframes[primary_record_set_id]
        df = df.dropna(subset=[numeric_field_id])
        # try numeric conversion (might produce errors if values are not all numbers)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].count() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):", filtered_df.shape[0])
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by another column if available
        candidate_group_fields = [c for c in cols if c != numeric_field_id and (('sample' in c.lower()) or ('type' in c.lower()) or ('group' in c.lower()))]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            print(f"Grouping by '{group_field_id}':")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped_df.head())
    else:
        print("No obvious numeric field found for EDA in this record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize distribution or relationships among data fields. We'll plot the chosen numeric field as a histogram if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id and sample_numeric_fields:
    df = dataframes[primary_record_set_id]
    numeric_field_id = sample_numeric_fields[0]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

- We loaded and explored the FAIR² 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' dataset using the Croissant schema and the `mlcroissant` library.
- The `@id` field was used throughout for consistent entity referencing and FAIR reproducibility.
- After listing available record sets, we extracted and processed data from one set, conducted basic EDA, and visualized numeric field distributions.

Further analysis can include more advanced processing and biological interpretation.